In [18]:
install.packages("parallel")

Warning message:
"package 'parallel' is a base package, and should not be updated"


In [12]:
library(simstudy)
library(lme4)
library(data.table)

defRE <- function(icc, dist = "binary", varW = NULL) {
  
  setVar <- iccRE(ICC = icc, dist = dist, varWithin = varW)
  def <- defData(varname = "a", formula = 0, variance = setVar, id = "cluster")
  
  return(def)
}

defBinOut <- function(p1, pctdelta) {
    
  p2 <- (1 - pctdelta) * p1
    
  int <- round(log( p1/(1-p1) ), 4)
  effect <- round(log( (p2/(1-p2)) / (p1/(1-p1) )), 4)
  formula <- genFormula( c(int, effect, 1), c("rx","a") )
    
  def <- defDataAdd(varname = "y", formula = formula, dist = "binary", 
                  link = "logit")
  return(def)
}

In [13]:
genDataSet <- function(nclust, clustsize, re.def, out.def) {
  
  dClust <- genData(nclust, re.def)
  dClust <- trtAssign(dClust, grpName = "rx")
  
  dPat <- genCluster(dtClust = dClust, cLevelVar = "cluster", 
                     numIndsVar = clustsize, level1ID = "id")
  dPat <- addColumns(out.def, dPat)
  
  return(dPat)
}

In [14]:
genBinEsts <- function(nclust, clustsize, re.def, out.def, fast = FALSE) {
  
  dP <- genDataSet(nclust, clustsize, re.def, out.def)

  mod.re <- glmer(y ~ rx + (1|cluster), data = dP, family = binomial,
      control = glmerControl( optimizer = "bobyqa", calc.derivs = !(fast) ))

  convStatus <- as.numeric(length(summary(mod.re)$optinfo$conv$lme4))
  
  res <- data.table(convStatus, re = VarCorr(mod.re)$cluster,
              t(coef(summary(mod.re))["rx",]))
  
  return(res)
}

In [15]:
(defa <- defRE(icc = 0.025))
##    varname formula variance   dist     link
## 1:       a       0   0.0844 normal identity
(defy <- defBinOut(0.40, 0.30))
##    varname                       formula variance   dist  link
## 1:       y -0.4055 + -0.539 * rx + 1 * a        0 binary logit

varname,formula,variance,dist,link
<chr>,<dbl>,<dbl>,<chr>,<chr>
a,0,0.08435559,normal,identity


varname,formula,variance,dist,link
<chr>,<chr>,<dbl>,<chr>,<chr>
y,-0.4055 + -0.539 * rx + 1 * a,0,binary,logit


In [16]:
RNGkind("L'Ecuyer-CMRG")
set.seed(2711)

genBinEsts(40, 10, defa, defy)
##    convStatus re.(Intercept) Estimate Std. Error z value Pr(>|z|)
## 1:          0          0.284   -0.625      0.279   -2.24   0.0249

convStatus,re.(Intercept),Estimate,Std. Error,z value,Pr(>|z|)
<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>
0,0.2836566,-0.6252467,0.2788404,-2.24231,0.02494132


In [19]:
library(parallel)

nIters <- 1000
results <- NULL

for (icc in ICC) {
  for (ss in SS) {
    for (nclust in nCLUST) {
      for (p1 in ctlPROB) {
        for (pdelta in pctDELTA) {
          
          clustsize <- ss %/% nclust
          p2 <- p1 * (1 - pdelta)
          
          defa <- defRE(icc)
          defy <- defBinOut(p1, pdelta)
          
          res <- rbindlist(mclapply(1:nIters, 
                        function(x) genBinEsts(nclust, clustsize, defa, defy)))
          
          dres <- data.table(icc, ss, nclust, clustsize, p1, p2, pdelta,
                             converged = res[, mean(convStatus == 0)],
                             p.conv = res[convStatus == 0, mean(`Pr(>|z|)` < 0.05)],
                             p.all = res[convStatus != 2, mean(`Pr(>|z|)` < 0.05)])
          
          print(dres)
          results <- rbind(results, dres)
          
        }
      }
    }
  }
}

ERROR: Error in eval(expr, envir, enclos): object 'ICC' not found
